In [89]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [90]:
FILE_PATH = Path.cwd().parent / "datasets" / "companies_messy_brazil.csv"

In [91]:
def load_dataframe(FILE_PATH):
    try:
        df = pd.read_csv(FILE_PATH)
        return df
    except FileNotFoundError :
        print(f' The desired dataset was not found at the given location please recheck address.')
    except Exception as e:
        print(f'Oops!!! Something unexpected occured  : \n {e}')


In [92]:
df = load_dataframe(FILE_PATH)

In [93]:
def examine_dataframe(df):
    print(f"Dataframe shape : {df.shape}")
    print("-"*100)
    print(f"Memory Usage : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print("-"*100)
    print(f"Column Dtypes: \n{df.dtypes.value_counts()}")
    print("-"*100)
    if df.isna().sum().sum() != 0 : 
        print(f"# Missing rows : {df.isna().sum().sum()}")
        print(f"Cols with missing rows : \n{df.isna().sum().sort_values(ascending=True).head(10)}")
    else:
        print(f"No missing rows")

In [94]:
examine_dataframe(df)

Dataframe shape : (141332, 6)
----------------------------------------------------------------------------------------------------
Memory Usage : 44.58 MB
----------------------------------------------------------------------------------------------------
Column Dtypes: 
object     4
int64      1
float64    1
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
No missing rows


In [95]:
# There are a lot of rows and we only have 6 columns
# Also we found out that the dataset has no missing rows
# This wont tell us anything and we need to further examine the unique values in each column

In [96]:
df.head()

,company_id,company_name,legal_nature,owner_qualification,capital_stock,company_size
0,41273639,MH MATERIAIS DE CONSTRUCAO LTDA,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,1000000.0,small-enterprise
1,41274138,CLINICA ESTETICA CAXIAS DO SUL RS LTDA,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,200000.0,micro-enterprise
2,41274505,G P CONSTRUCOES E SERVICOS LTDA,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,500000.0,small-enterprise
3,41274745,UNICREDIT BANK SAO PAULO CONSULTORIA UNIPESSOA...,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,159600.0,small-enterprise
4,41274856,PRODUCON PRODUTOS PARA CONSTRUCAO LTDA,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,200000.0,micro-enterprise


In [97]:
# TO look for number of unique values in each column :
df.nunique()

company_id             141332
company_name           140739
legal_nature               22
owner_qualification        13
capital_stock           22404
company_size                3
dtype: int64

In [98]:
# As company_id and company_name have almost each value unique I will drop them for now
df = df.drop(columns=['company_name','company_id'])

In [99]:
df.head()

,legal_nature,owner_qualification,capital_stock,company_size
0,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,1000000.0,small-enterprise
1,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,200000.0,micro-enterprise
2,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,500000.0,small-enterprise
3,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,159600.0,small-enterprise
4,Limited Liability Business Company (LLC),Managing Partner / Partner-Administrator,200000.0,micro-enterprise


In [100]:
# Capital size is already in float so we have to handle only 3 columns

In [101]:
# We will simply use OneHotEncoding here while encoding and also the weight balance is somewhat imbalanced but not that much
df['company_size'].value_counts()

company_size
micro-enterprise    66202
other               42520
small-enterprise    32610
Name: count, dtype: int64

In [102]:
df['legal_nature'].value_counts()

legal_nature
Limited Liability Business Company (LLC)           119288
Sole Proprietorship                                 15209
Privately Held Corporation                           2897
Silent Partnership                                   2585
Simple Limited Partnership                            550
Individual Limited Liability Company (Business)       182
Simple Innovation Company                             144
Sole Member Law Firm                                  123
Simple Partnership (Pure)                             114
Cooperative                                            78
Publicly Traded Corporation                            52
Individual Real Estate Company                         50
Mixed-Capital Company                                  21
State-Owned Enterprise                                 15
General Partnership                                     8
Brazilian Branch of a Foreign Company                   5
Private Association                                     4
S

In [103]:
# For this much number of values we will map them into groups and then encode them in the group they fall in
legal_group_map = {
    "Limited Liability Business Company (LLC)": "Corporate_Limited_Liability",
    "Privately Held Corporation": "Corporate_Limited_Liability",
    "Publicly Traded Corporation": "Corporate_Limited_Liability",
    "Partnership Limited by Shares": "Corporate_Limited_Liability",
    
    "Sole Proprietorship": "Sole_Proprietorship_Individual",
    "Individual Limited Liability Company (Business)": "Sole_Proprietorship_Individual",
    "Sole Member Law Firm": "Sole_Proprietorship_Individual",
    "Individual Real Estate Company": "Sole_Proprietorship_Individual",
    "Individual Limited Liability Company (Simple)": "Sole_Proprietorship_Individual",
    
    "Silent Partnership": "Partnerships",
    "Simple Limited Partnership": "Partnerships",
    "Simple Partnership (Pure)": "Partnerships",
    "General Partnership": "Partnerships",
    "Simple General Partnership": "Partnerships",
    
    "Mixed-Capital Company": "State_Public_Sector",
    "State-Owned Enterprise": "State_Public_Sector",
    "Autonomous Social Service": "State_Public_Sector",
    
    "Cooperative": "Cooperatives_Non_Profits",
    "Consumer Cooperatives": "Cooperatives_Non_Profits",
    "Private Association": "Cooperatives_Non_Profits",
    
    "Simple Innovation Company": "Specialized_Foreign",
    "Brazilian Branch of a Foreign Company": "Specialized_Foreign"
}



In [104]:
# Now for the last column owner_qualification
df['owner_qualification'].value_counts()

owner_qualification
Managing Partner / Partner-Administrator                         107027
Administrator / Manager                                           15236
Entrepreneur / Business Owner                                     15201
Director / Officer                                                 1634
President / Chair                                                  1343
Beneficial Owner (individual) resident or domiciled in Brazil       442
Judicial Administrator (Court-appointed)                            302
Sole Owner of an Individual Real Estate Company                      50
Ostensible Partner (Managing partner in a silent partnership)        32
Liquidator                                                           29
Executor / Estate Administrator                                      22
Attorney-in-fact / Legal Representative (Power of Attorney)          13
Intervenor / Court-appointed Administrator                            1
Name: count, dtype: int64

In [105]:
# We will also make new categoris categoris for this to reduce cardinality
owner_map = {
    # 1. Executive Management
    "Managing Partner / Partner-Administrator": "Executive",
    "Administrator / Manager": "Executive",
    "Director / Officer": "Executive",
    "President / Chair": "Executive",
    
    # 2. Business Owners
    "Entrepreneur / Business Owner": "Owner",
    "Beneficial Owner (individual) resident or domiciled in Brazil": "Owner",
    "Sole Owner of an Individual Real Estate Company": "Owner",
    "Ostensible Partner (Managing partner in a silent partnership)": "Owner",
    
    # 3. Court Appointed / Legal Fiduciaries
    "Judicial Administrator (Court-appointed)": "Legal_Fiduciary",
    "Liquidator": "Legal_Fiduciary",
    "Executor / Estate Administrator": "Legal_Fiduciary",
    "Attorney-in-fact / Legal Representative (Power of Attorney)": "Legal_Fiduciary",
    "Intervenor / Court-appointed Administrator": "Legal_Fiduciary"
}


In [106]:
#  Map your existing column to the new grouped categories

df['legal_nature'] = df['legal_nature'].map(legal_group_map)

df['owner_qualification'] = df['owner_qualification'].map(owner_map)

In [107]:
df.head()

,legal_nature,owner_qualification,capital_stock,company_size
0,Corporate_Limited_Liability,Executive,1000000.0,small-enterprise
1,Corporate_Limited_Liability,Executive,200000.0,micro-enterprise
2,Corporate_Limited_Liability,Executive,500000.0,small-enterprise
3,Corporate_Limited_Liability,Executive,159600.0,small-enterprise
4,Corporate_Limited_Liability,Executive,200000.0,micro-enterprise


In [109]:
df.nunique() # Way less cardinality for object cols 

legal_nature               6
owner_qualification        3
capital_stock          22404
company_size               3
dtype: int64

In [ ]:
# Now using get_dummies on these cols to OHC them using drop first
pd.get_dummies(
                data = df,
                columns = df.select_dtypes('object').columns,
                dtype=int,
                drop_first=True # For linear models
)

,capital_stock,legal_nature_Cooperatives_Non_Profits,legal_nature_Corporate_Limited_Liability,legal_nature_Partnerships,legal_nature_Sole_Proprietorship_Individual,legal_nature_Specialized_Foreign,legal_nature_State_Public_Sector,owner_qualification_Executive,owner_qualification_Legal_Fiduciary,owner_qualification_Owner,company_size_micro-enterprise,company_size_other,company_size_small-enterprise
0,1000000.0,0,1,0,0,0,0,1,0,0,0,0,1
1,200000.0,0,1,0,0,0,0,1,0,0,1,0,0
2,500000.0,0,1,0,0,0,0,1,0,0,0,0,1
3,159600.0,0,1,0,0,0,0,1,0,0,0,0,1
4,200000.0,0,1,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
141327,800000.0,0,1,0,0,0,0,1,0,0,1,0,0
141328,1000000.0,0,1,0,0,0,0,1,0,0,0,1,0
141329,200000.0,0,1,0,0,0,0,1,0,0,0,1,0
141330,1500000.0,0,1,0,0,0,0,1,0,0,0,1,0


In [115]:
# A quick check to see whether all the column in capital_stock are float or they are int in float format
(df['capital_stock'] % 1 == 0).all() # Some values are genuine float

np.False_